# Notebook 03 — Phase-Lock Correction Loop

**Repo:** `residual-phase-lock`  
**Notebook:** `03_phase_lock_correction_loop.ipynb`

## Claim

> Residual phase-lock stabilizes coherence against drift.

Notebook arc:

```text
01 → residual reveals structure
02 → topology drift appears
03 → phase-lock correction stabilizes coherence
```

This notebook uses the same parity setting as Notebook 02, but adds a correction loop that measures residual drift and applies a simple global-structure repair step.

## 1. Setup

This notebook uses the repo helper:

```python
from src.export import ExportManager
```

It saves numbered artifacts into:

```text
figures/
results/
docs/
```

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# Works in GitHub Actions from repo root.
# Works in Colab if notebook is opened from the GitHub repo.
# Fallback helps if Colab starts inside notebooks/.
if os.path.exists("../src"):
    sys.path.append("..")

try:
    from src.export import ExportManager
except ModuleNotFoundError:
    # Minimal local fallback so the notebook still runs before src/export.py exists.
    class ExportManager:
        def __init__(self, notebook_id, notebook_slug):
            self.id = notebook_id
            self.slug = notebook_slug
            self.fig_dir = "figures"
            self.results_dir = "results"
            self.docs_dir = "docs"
            os.makedirs(self.fig_dir, exist_ok=True)
            os.makedirs(self.results_dir, exist_ok=True)
            os.makedirs(self.docs_dir, exist_ok=True)

        def save_fig(self, name):
            path = f"{self.fig_dir}/{self.id}_{name}.png"
            plt.savefig(path, dpi=220, bbox_inches="tight")
            print(f"[export:fallback] saved figure: {path}")

        def save_csv(self, df, name):
            path = f"{self.results_dir}/{self.id}_{name}.csv"
            df.to_csv(path, index=False)
            print(f"[export:fallback] saved csv: {path}")

        def save_json(self, obj, name):
            path = f"{self.results_dir}/{self.id}_{name}.json"
            with open(path, "w") as f:
                json.dump(obj, f, indent=2)
            print(f"[export:fallback] saved json: {path}")

        def write_md(self, title, metrics_dict, figure_names, interpretation=None):
            md_path = f"{self.docs_dir}/{self.id}_{self.slug}.md"
            metrics_lines = "\n".join([f"| {k} | {v:.3f} |" for k, v in metrics_dict.items()])
            figure_lines = "\n\n".join([f"![{name}](../figures/{self.id}_{name}.png)" for name in figure_names])
            interpretation_block = ""
            if interpretation:
                interpretation_block = f'''
## Interpretation

```text
{interpretation.strip()}
```
'''
            md = f'''# Notebook {self.id} — {title}

## Results

| Metric | Value |
|--------|------:|
{metrics_lines}

## Figures

{figure_lines}

{interpretation_block}
'''
            with open(md_path, "w") as f:
                f.write(md)
            print(f"[export:fallback] saved markdown: {md_path}")

np.random.seed(44)

NOTEBOOK_ID = "03"
NOTEBOOK_SLUG = "phase_lock_correction_loop"

exp = ExportManager(NOTEBOOK_ID, NOTEBOOK_SLUG)

## 2. Generate global-structure data

We reuse the parity task:

```text
label = sum(bits) mod 2
```

Parity acts as the known global structure. A model output that violates parity has drifted from the constraint.

In [ ]:
def make_parity_data(n_samples=5000, dim=16, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.integers(0, 2, size=(n_samples, dim))
    y = X.sum(axis=1) % 2
    return X, y

DIM = 16

X_train, y_train = make_parity_data(n_samples=2500, dim=DIM, seed=3)
X_test, y_test = make_parity_data(n_samples=5000, dim=DIM, seed=4)

test_df = pd.DataFrame(X_test, columns=[f"bit_{i}" for i in range(DIM)])
test_df["true_parity"] = y_test

# Raw synthetic data is saved for completeness, but you may choose not to commit it.
exp.save_csv(test_df, "test_parity_data")

test_df.head()

## 3. Train baseline model

The baseline model provides local fit but does not explicitly preserve the global parity constraint.

In [ ]:
model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=500,
    random_state=44,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=25,
)

model.fit(X_train, y_train)

baseline_pred = model.predict(X_test)
baseline_accuracy = float(accuracy_score(y_test, baseline_pred))
baseline_drift = (baseline_pred != y_test).astype(int)
baseline_drift_rate = float(baseline_drift.mean())
baseline_residual = y_test - baseline_pred

print(f"Baseline accuracy:   {baseline_accuracy:.4f}")
print(f"Baseline drift rate: {baseline_drift_rate:.4f}")

## 4. Define a phase-lock correction loop

For this controlled task, we know the global structure exactly.  
The correction loop therefore has three steps:

```text
predict → detect drift → revise toward global structure
```

This is intentionally minimal. The point is not to claim this parity repair is a general transformer architecture. The point is to isolate the mechanism:

```text
residual → drift signal
phase-lock → correction
coherence → restored structure
```

In [ ]:
def parity_constraint(x):
    """Known global structure: parity of the full binary vector."""
    return x.sum(axis=1) % 2

def detect_drift(pred, constraint_label):
    """Return drift mask and residual under known global constraint."""
    drift = (pred != constraint_label).astype(int)
    residual = constraint_label - pred
    return drift, residual

def phase_lock_correct(pred, constraint_label, correction_strength=1.0):
    """
    Minimal phase-lock correction.

    correction_strength = 0.0 keeps the original prediction.
    correction_strength = 1.0 fully locks to the known global constraint.
    Values in between simulate partial correction.
    """
    corrected = pred.copy()
    drift, residual = detect_drift(pred, constraint_label)

    # Deterministic partial correction: correct a fixed fraction of drifted outputs.
    drift_indices = np.where(drift == 1)[0]
    n_correct = int(np.round(correction_strength * len(drift_indices)))

    if n_correct > 0:
        corrected_indices = drift_indices[:n_correct]
        corrected[corrected_indices] = constraint_label[corrected_indices]

    return corrected, drift, residual

constraint_label = parity_constraint(X_test)

# Sanity check: constraint label should equal y_test for this controlled task.
assert np.array_equal(constraint_label, y_test)

## 5. Sweep correction strength

We test a range of correction strengths from no correction to full phase-lock.

In [ ]:
strengths = np.linspace(0, 1, 11)

sweep_rows = []
predictions_by_strength = {}

for strength in strengths:
    corrected_pred, drift_mask, residual = phase_lock_correct(
        baseline_pred,
        constraint_label,
        correction_strength=float(strength),
    )

    acc = float(accuracy_score(y_test, corrected_pred))
    drift_rate = float((corrected_pred != y_test).mean())
    residual_norm = float(np.linalg.norm(y_test - corrected_pred))
    coherence_score = float(1.0 - drift_rate)

    predictions_by_strength[float(strength)] = corrected_pred

    sweep_rows.append({
        "correction_strength": float(strength),
        "accuracy": acc,
        "drift_rate": drift_rate,
        "residual_norm": residual_norm,
        "coherence_score": coherence_score,
    })

sweep_df = pd.DataFrame(sweep_rows)
exp.save_csv(sweep_df, "phase_lock_sweep")

sweep_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(sweep_df["correction_strength"], sweep_df["drift_rate"], marker="o", label="drift rate")
plt.plot(sweep_df["correction_strength"], sweep_df["coherence_score"], marker="o", label="coherence score")
plt.xlabel("correction strength")
plt.ylabel("rate")
plt.title("Phase-lock correction: drift decreases as coherence stabilizes")
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
exp.save_fig("phase_lock_sweep")
plt.show()

## 6. Before / after correction

We compare baseline output to full phase-lock correction.

In [ ]:
corrected_pred, detected_drift, detected_residual = phase_lock_correct(
    baseline_pred,
    constraint_label,
    correction_strength=1.0,
)

corrected_accuracy = float(accuracy_score(y_test, corrected_pred))
corrected_drift_rate = float((corrected_pred != y_test).mean())
corrected_residual = y_test - corrected_pred

drift_reduction = float(baseline_drift_rate - corrected_drift_rate)
relative_drift_reduction = float(drift_reduction / baseline_drift_rate) if baseline_drift_rate > 0 else 0.0

print(f"Corrected accuracy:   {corrected_accuracy:.4f}")
print(f"Corrected drift rate: {corrected_drift_rate:.4f}")
print(f"Drift reduction:      {drift_reduction:.4f}")
print(f"Relative reduction:   {relative_drift_reduction:.4f}")

In [ ]:
before_after = pd.DataFrame({
    "condition": ["baseline", "phase_lock_corrected"],
    "accuracy": [baseline_accuracy, corrected_accuracy],
    "drift_rate": [baseline_drift_rate, corrected_drift_rate],
    "residual_norm": [
        float(np.linalg.norm(baseline_residual)),
        float(np.linalg.norm(corrected_residual)),
    ],
    "coherence_score": [
        1.0 - baseline_drift_rate,
        1.0 - corrected_drift_rate,
    ],
})

exp.save_csv(before_after, "before_after")
before_after

In [ ]:
plt.figure(figsize=(7, 4))
x_pos = np.arange(len(before_after))
width = 0.35

plt.bar(x_pos - width/2, before_after["drift_rate"], width, label="drift rate")
plt.bar(x_pos + width/2, before_after["coherence_score"], width, label="coherence score")

plt.xticks(x_pos, before_after["condition"], rotation=15, ha="right")
plt.ylabel("rate")
plt.title("Before / after phase-lock correction")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
exp.save_fig("before_after_phase_lock")
plt.show()

## 7. Residual distribution before and after

Residuals show the correction effect directly.

In [ ]:
residual_compare = pd.DataFrame({
    "baseline_residual": baseline_residual,
    "corrected_residual": corrected_residual,
})

baseline_counts = (
    pd.Series(baseline_residual)
    .value_counts()
    .sort_index()
    .rename_axis("residual")
    .reset_index(name="baseline_count")
)

corrected_counts = (
    pd.Series(corrected_residual)
    .value_counts()
    .sort_index()
    .rename_axis("residual")
    .reset_index(name="corrected_count")
)

residual_counts = pd.merge(
    baseline_counts,
    corrected_counts,
    on="residual",
    how="outer"
).fillna(0)

exp.save_csv(residual_counts, "residual_counts_before_after")

residual_counts

In [ ]:
residual_values = residual_counts["residual"].astype(str)
idx = np.arange(len(residual_values))
width = 0.35

plt.figure(figsize=(7, 4))
plt.bar(idx - width/2, residual_counts["baseline_count"], width, label="baseline")
plt.bar(idx + width/2, residual_counts["corrected_count"], width, label="corrected")
plt.xticks(idx, residual_values)
plt.xlabel("residual")
plt.ylabel("count")
plt.title("Residual distribution before and after phase-lock")
plt.legend()
plt.tight_layout()
exp.save_fig("residual_distribution_before_after")
plt.show()

## 8. Drift by Hamming weight before and after

This shows whether correction stabilizes drift across the structure groups used in Notebook 02.

In [ ]:
drift_by_weight_df = pd.DataFrame({
    "hamming_weight": X_test.sum(axis=1),
    "baseline_drift": baseline_drift,
    "corrected_drift": (corrected_pred != y_test).astype(int),
})

drift_by_weight = (
    drift_by_weight_df.groupby("hamming_weight")
    .agg(
        count=("baseline_drift", "size"),
        baseline_drift_rate=("baseline_drift", "mean"),
        corrected_drift_rate=("corrected_drift", "mean"),
    )
    .reset_index()
)

exp.save_csv(drift_by_weight, "drift_by_hamming_weight_before_after")
drift_by_weight

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(
    drift_by_weight["hamming_weight"],
    drift_by_weight["baseline_drift_rate"],
    marker="o",
    label="baseline drift",
)
plt.plot(
    drift_by_weight["hamming_weight"],
    drift_by_weight["corrected_drift_rate"],
    marker="o",
    label="corrected drift",
)
plt.ylim(0, 1)
plt.xlabel("Hamming weight")
plt.ylabel("drift rate")
plt.title("Topology drift by Hamming weight before and after phase-lock")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
exp.save_fig("drift_by_hamming_weight_before_after")
plt.show()

## 9. Summary outputs

The summary table is saved into `results/03_summary.csv` and `results/03_summary.json`.

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "baseline_accuracy",
        "baseline_drift_rate",
        "corrected_accuracy",
        "corrected_drift_rate",
        "drift_reduction",
        "relative_drift_reduction",
        "baseline_coherence_score",
        "corrected_coherence_score",
    ],
    "value": [
        baseline_accuracy,
        baseline_drift_rate,
        corrected_accuracy,
        corrected_drift_rate,
        drift_reduction,
        relative_drift_reduction,
        1.0 - baseline_drift_rate,
        1.0 - corrected_drift_rate,
    ],
})

exp.save_csv(summary, "summary")
summary_json = {row["metric"]: float(row["value"]) for _, row in summary.iterrows()}
exp.save_json(summary_json, "summary")

summary

## 10. Generate markdown summary

This writes:

```text
docs/03_phase_lock_correction_loop.md
```

The markdown file points to repo-local figures and summarizes the key metrics.

In [ ]:
exp.write_md(
    title="Phase-Lock Correction Loop",
    metrics_dict={
        "Baseline accuracy": baseline_accuracy,
        "Baseline drift rate": baseline_drift_rate,
        "Corrected accuracy": corrected_accuracy,
        "Corrected drift rate": corrected_drift_rate,
        "Relative drift reduction": relative_drift_reduction,
        "Corrected coherence score": 1.0 - corrected_drift_rate,
    },
    figure_names=[
        "phase_lock_sweep",
        "before_after_phase_lock",
        "residual_distribution_before_after",
        "drift_by_hamming_weight_before_after",
    ],
    interpretation="""
residual → drift signal
phase-lock → correction loop
coherence stabilizes against drift
""",
)

## 11. Output export

Run this final cell in Colab to download notebook outputs.

It creates:

```text
03_phase_lock_correction_loop_outputs.zip
├── figures/
├── results/
└── docs/
```

In [ ]:
# =========================
# Export outputs (Colab)
# =========================

ZIP_NAME = "03_phase_lock_correction_loop_outputs.zip"

os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("docs", exist_ok=True)

!zip -r $ZIP_NAME figures results docs

try:
    from google.colab import files
    files.download(ZIP_NAME)
except ImportError:
    print(f"Not running in Colab. Output zip created locally: {ZIP_NAME}")

## 12. Takeaway

This notebook supports the third repo claim:

```text
residuals expose drift
phase-lock corrects drift
coherence stabilizes
```

Notebook arc:

```text
01 → residual reveals structure
02 → topology drift appears
03 → phase-lock stabilizes coherence against drift
```